# Starbucks Menu EDA

Exploratory data analysis of the Starbucks beverage menu dataset.

**Data source**: Public nutrition dataset (reisanar/datasets on GitHub), augmented with statistically generated price and seasonal flag columns.

**Analysis goals**:
1. Category distribution
2. Price distribution by category and size
3. Calories vs. Price relationship
4. Seasonal item trends

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from pathlib import Path
matplotlib.rcParams["figure.figsize"] = (12, 6)
matplotlib.rcParams["figure.dpi"] = 100

ON_KAGGLE = Path("/kaggle/working").exists()
if ON_KAGGLE:
    DATA_DIR = Path("/kaggle/input/starbucks-recommendation-engine")
else:
    DATA_DIR = Path("../data/raw")

df = pd.read_csv(DATA_DIR / "starbucks_menu.csv")
print(f"Shape: {df.shape}")
df.head(10)

In [ ]:
# Basic statistics
df.describe()

In [ ]:
# Missing values check
df.isnull().sum()

## 1. Category Distribution

In [ ]:
cat_counts = df["category"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart
cat_counts.plot(kind="barh", ax=axes[0], color="seagreen")
axes[0].set_title("Number of Items by Category")
axes[0].set_xlabel("Count")

# Pie chart
cat_counts.plot(kind="pie", ax=axes[1], autopct="%1.1f%%", startangle=90)
axes[1].set_title("Category Proportion")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

## 2. Price Distribution by Category and Size

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Box plot: price by category
df.boxplot(column="price_usd", by="category", ax=axes[0], vert=False)
axes[0].set_title("Price Distribution by Category")
axes[0].set_xlabel("Price (USD)")
plt.sca(axes[0])
plt.xticks(rotation=0)

# Box plot: price by size
size_order = ["Short", "Tall", "Grande", "Venti"]
df["size"] = pd.Categorical(df["size"], categories=size_order, ordered=True)
df.boxplot(column="price_usd", by="size", ax=axes[1])
axes[1].set_title("Price Distribution by Size")
axes[1].set_ylabel("Price (USD)")

plt.suptitle("")
plt.tight_layout()
plt.show()

# Price statistics by category
df.groupby("category")["price_usd"].agg(["mean", "median", "std"]).round(2)

## 3. Calories vs. Price Relationship

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

categories = df["category"].unique()
colors = plt.cm.Set2(np.linspace(0, 1, len(categories)))

for cat, color in zip(categories, colors):
    mask = df["category"] == cat
    ax.scatter(
        df.loc[mask, "calories"],
        df.loc[mask, "price_usd"],
        label=cat, alpha=0.7, s=50, color=color
    )

# Trend line
valid = df.dropna(subset=["calories", "price_usd"])
z = np.polyfit(valid["calories"], valid["price_usd"], 1)
p = np.poly1d(z)
x_line = np.linspace(valid["calories"].min(), valid["calories"].max(), 100)
ax.plot(x_line, p(x_line), "--", color="gray", alpha=0.8, label=f"Trend (slope={z[0]:.4f})")

ax.set_xlabel("Calories")
ax.set_ylabel("Price (USD)")
ax.set_title("Calories vs. Price by Category")
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

# Correlation
corr = valid[["calories", "price_usd", "sugar_g", "caffeine_mg"]].corr()
print("Correlation matrix:")
corr.round(3)

## 4. Seasonal Item Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Seasonal vs regular count by category
seasonal_cat = df.groupby(["category", "seasonal_flag"]).size().unstack(fill_value=0)
seasonal_cat.columns = ["Regular", "Seasonal"]
seasonal_cat.plot(kind="barh", stacked=True, ax=axes[0], color=["steelblue", "coral"])
axes[0].set_title("Seasonal vs Regular by Category")
axes[0].set_xlabel("Count")

# Price comparison: seasonal vs regular
df["type"] = df["seasonal_flag"].map({0: "Regular", 1: "Seasonal"})
df.boxplot(column="price_usd", by="type", ax=axes[1])
axes[1].set_title("Price: Seasonal vs Regular")
axes[1].set_ylabel("Price (USD)")
plt.sca(axes[1])
plt.xlabel("")

# Calorie comparison: seasonal vs regular
df.boxplot(column="calories", by="type", ax=axes[2])
axes[2].set_title("Calories: Seasonal vs Regular")
axes[2].set_ylabel("Calories")
plt.sca(axes[2])
plt.xlabel("")

plt.suptitle("")
plt.tight_layout()
plt.show()

# Summary stats
print("Seasonal item summary:")
df.groupby("type")[["price_usd", "calories", "sugar_g", "caffeine_mg"]].mean().round(2)

## 5. Nutrition Profile Heatmap

In [ ]:
# Average nutrition profile by category
nutrition_cols = ["price_usd", "calories", "sugar_g", "caffeine_mg"]
profile = df.groupby("category")[nutrition_cols].mean()

# Normalize for heatmap
profile_norm = (profile - profile.min()) / (profile.max() - profile.min())

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(profile_norm.values, cmap="YlOrRd", aspect="auto")

ax.set_xticks(range(len(nutrition_cols)))
ax.set_xticklabels(nutrition_cols, rotation=45, ha="right")
ax.set_yticks(range(len(profile.index)))
ax.set_yticklabels(profile.index)

# Add value annotations
for i in range(len(profile.index)):
    for j in range(len(nutrition_cols)):
        ax.text(j, i, f"{profile.values[i, j]:.1f}",
                ha="center", va="center", fontsize=8)

plt.colorbar(im, label="Normalized value")
ax.set_title("Average Nutrition Profile by Category (normalized heatmap, raw values shown)")
plt.tight_layout()
plt.show()